In [1]:
import json
import time
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from sentence_transformers import SentenceTransformer

PROCESSED_FILE = Path("../data/processed/cicids_processed.csv")
OUTPUT_DIR = Path("../data/embeddings")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_FILE = OUTPUT_DIR / "cicids_sample.csv"
EMBED_FILE = OUTPUT_DIR / "cicids_embeddings.npy"
META_FILE = OUTPUT_DIR / "cicids_embedding_meta.json"

MODEL_NAME = "all-MiniLM-L6-v2"
RANDOM_SEED = 42
SAMPLE_PER_TACTIC = 2000
BENIGN_SAMPLE = 2000
BATCH_SIZE = 128


In [2]:
# Load the processed data
df = pd.read_csv(PROCESSED_FILE, index_col="alert_id", low_memory=False)
print(f"Loaded {len(df):,} rows")

Loaded 435,290 rows


In [3]:
# build a balanced sample of attack and benign rows for embedding
attack_parts = []

attack_df = df[df["attck_tactic"] != "Benign"].copy()
tactics = sorted(attack_df["attck_tactic"].dropna().unique())

print("\nSampling attack rows by tactic:")
for tactic in tactics:
    tactic_df = attack_df[attack_df["attck_tactic"] == tactic]
    n = min(len(tactic_df), SAMPLE_PER_TACTIC)
    sampled = tactic_df.sample(n=n, random_state=RANDOM_SEED)
    attack_parts.append(sampled)
    print(f"  {tactic}: {len(tactic_df):,} available -> {n:,} sampled")

benign_df = df[df["attck_tactic"] == "Benign"].copy()
n_benign = min(len(benign_df), BENIGN_SAMPLE)
benign_sample = benign_df.sample(n=n_benign, random_state=RANDOM_SEED)

print(f"  Benign: {len(benign_df):,} available -> {n_benign:,} sampled")


Sampling attack rows by tactic:
  Command And Control: 2,002 available -> 2,000 sampled
  Credential Access: 15,342 available -> 2,000 sampled
  Discovery: 158,930 available -> 2,000 sampled
  Execution: 652 available -> 652 sampled
  Impact: 128,027 available -> 2,000 sampled
  Initial Access: 21 available -> 21 sampled
  Benign: 130,316 available -> 2,000 sampled


In [5]:
# Combine and shuffle the sample
sample = pd.concat(attack_parts + [benign_sample], ignore_index=False)

#preserve original accesed row id
sample["alert_id"] = sample.index

# Shuffle the sample and reset the index to create a new sample_id
sample = sample.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
sample.index.name = "sample_id"
sample["sample_id"] = sample.index

missing_texts = sample["alert_text"].isna().sum()
print(f"\nMissing alert_text rows: {missing_texts:,}")
sample = sample[sample["alert_text"].notna()].copy()

print(f"\nFinal sample size: {len(sample):,}")
print(sample["attck_tactic"].value_counts().to_string())

sample.to_csv(SAMPLE_FILE, index=True)
print(f"\nSaved sample to {SAMPLE_FILE}")

print("\nSample summary:")
sample[["sample_id", "alert_id", "Label", "attck_tactic", "attck_technique_id"]].head()


Missing alert_text rows: 0

Final sample size: 10,673
attck_tactic
Discovery              2000
Credential Access      2000
Benign                 2000
Command And Control    2000
Impact                 2000
Execution               652
Initial Access           21

Saved sample to ../data/embeddings/cicids_sample.csv

Sample summary:


,sample_id,alert_id,Label,attck_tactic,attck_technique_id
sample_id,,,,,
0,0,164944,PortScan,Discovery,T1046
1,1,386077,FTP Patator,Credential Access,T1110.001
2,2,331474,BENIGN,Benign,BENIGN
3,3,386046,FTP Patator,Credential Access,T1110.001
4,4,405773,BENIGN,Benign,BENIGN


In [6]:
# Load the embedding model
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = SentenceTransformer(MODEL_NAME, device=device)
print(f"Loaded model: {MODEL_NAME}")
print(f"Embedding size: {model.get_sentence_embedding_dimension()}")

Using device: cuda


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded model: all-MiniLM-L6-v2
Embedding size: 384


In [7]:
# create embeddings 
texts = sample["alert_text"].tolist()

print(f"Embedding {len(texts):,} alerts")
print(f"Batch size: {BATCH_SIZE}")
print(f"Duplicate texts: {sample['alert_text'].duplicated().sum():,}")

start = time.time()

embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

elapsed = time.time() - start

print(f"Embedding complete")
print(f"Time: {elapsed:.1f} seconds")
print(f"Shape: {embeddings.shape}")
print(f"Vector norm example: {np.linalg.norm(embeddings[0]):.4f}")

Embedding 10,673 alerts
Batch size: 128
Duplicate texts: 773


Batches:   0%|          | 0/84 [00:00<?, ?it/s]

Embedding complete
Time: 5.1 seconds
Shape: (10673, 384)
Vector norm example: 1.0000


In [8]:
# Save the embeddings and metadata
np.save(EMBED_FILE, embeddings)
print(f"\nSaved embeddings to {EMBED_FILE}")

meta = {
    "model_name": MODEL_NAME,
    "device": device,
    "n_rows": int(len(sample)),
    "embedding_dim": int(embeddings.shape[1]),
    "embedding_dtype": str(embeddings.dtype),
    "sample_per_tactic": SAMPLE_PER_TACTIC,
    "benign_sample": BENIGN_SAMPLE,
    "random_seed": RANDOM_SEED,
    "batch_size": BATCH_SIZE,
    "normalized_embeddings": True,
    "tactic_counts": sample["attck_tactic"].value_counts().to_dict(),
    "technique_counts": sample["attck_technique_id"].value_counts().to_dict(),
}

with open(META_FILE, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

print(f"Saved metadata to {META_FILE}")


Saved embeddings to ../data/embeddings/cicids_embeddings.npy
Saved metadata to ../data/embeddings/cicids_embedding_meta.json
